# 12 — Model Pretrained Trainable / 预训练模型

**Chapter 22 — File 12 of 15 / 第22章 — 第12个文件（共15个）**

---

## Summary / 总结

This script demonstrates **vgg16 transfer learning on the planet dataset with some trainable layers**.

本脚本演示 **vgg16 transfer learning on the planet dataset with some trainable layers**。

---
## Step 1 — vgg16 transfer learning on the planet dataset with some trainable layers

In [ ]:
import sys
from numpy import load
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from keras import backend
from keras.layers import Dense
from keras.layers import Flatten
from keras.optimizers import SGD
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.preprocessing.image import ImageDataGenerator

---
## Step 2 — load train and test dataset

In [ ]:
def load_dataset():

---
## Step 3 — load dataset

In [ ]:
data = load('planet_data.npz')
	X, y = data['arr_0'], data['arr_1']

---
## Step 4 — separate into train and test datasets

In [ ]:
trainX, testX, trainY, testY = train_test_split(X, y, test_size=0.3, random_state=1)
	print(trainX.shape, trainY.shape, testX.shape, testY.shape)
	return trainX, trainY, testX, testY

---
## Step 5 — calculate fbeta score for multi-class/label classification

In [ ]:
def fbeta(y_true, y_pred, beta=2):

---
## Step 6 — clip predictions

In [ ]:
y_pred = backend.clip(y_pred, 0, 1)

---
## Step 7 — calculate elements

In [ ]:
tp = backend.sum(backend.round(backend.clip(y_true * y_pred, 0, 1)), axis=1)
	fp = backend.sum(backend.round(backend.clip(y_pred - y_true, 0, 1)), axis=1)
	fn = backend.sum(backend.round(backend.clip(y_true - y_pred, 0, 1)), axis=1)

---
## Step 8 — calculate precision

In [ ]:
p = tp / (tp + fp + backend.epsilon())

---
## Step 9 — calculate recall

In [ ]:
r = tp / (tp + fn + backend.epsilon())

---
## Step 10 — calculate fbeta, averaged across each class

In [ ]:
bb = beta ** 2
	fbeta_score = backend.mean((1 + bb) * (p * r) / (bb * p + r + backend.epsilon()))
	return fbeta_score

---
## Step 11 — define cnn model

In [ ]:
def define_model(in_shape=(128, 128, 3), out_shape=17):

---
## Step 12 — load model

In [ ]:
model = VGG16(include_top=False, input_shape=in_shape)

---
## Step 13 — mark loaded layers as not trainable

In [ ]:
for layer in model.layers:
		layer.trainable = False

---
## Step 14 — allow last vgg block to be trainable

In [ ]:
model.get_layer('block5_conv1').trainable = True
	model.get_layer('block5_conv2').trainable = True
	model.get_layer('block5_conv3').trainable = True
	model.get_layer('block5_pool').trainable = True

---
## Step 15 — add new classifier layers

In [ ]:
flat1 = Flatten()(model.layers[-1].output)
	class1 = Dense(128, activation='relu', kernel_initializer='he_uniform')(flat1)
	output = Dense(out_shape, activation='sigmoid')(class1)

---
## Step 16 — define new model

In [ ]:
model = Model(inputs=model.inputs, outputs=output)

---
## Step 17 — compile model

In [ ]:
opt = SGD(lr=0.01, momentum=0.9)
	model.compile(optimizer=opt, loss='binary_crossentropy', metrics=[fbeta])
	return model

---
## Step 18 — plot diagnostic learning curves

In [ ]:
def summarize_diagnostics(history):

---
## Step 19 — plot loss

In [ ]:
pyplot.subplot(211)
	pyplot.title('Cross Entropy Loss')
	pyplot.plot(history.history['loss'], color='blue', label='train')
	pyplot.plot(history.history['val_loss'], color='orange', label='test')

---
## Step 20 — plot accuracy

In [ ]:
pyplot.subplot(212)
	pyplot.title('Fbeta')
	pyplot.plot(history.history['fbeta'], color='blue', label='train')
	pyplot.plot(history.history['val_fbeta'], color='orange', label='test')

---
## Step 21 — save plot to file

In [ ]:
filename = sys.argv[0].split('/')[-1]
	pyplot.savefig(filename + '_plot.png')
	pyplot.close()

---
## Step 22 — run the test harness for evaluating a model

In [ ]:
def run_test_harness():

---
## Step 23 — load dataset

In [ ]:
trainX, trainY, testX, testY = load_dataset()

---
## Step 24 — create data generator

In [ ]:
datagen = ImageDataGenerator(featurewise_center=True)

---
## Step 25 — specify imagenet mean values for centering

In [ ]:
datagen.mean = [123.68, 116.779, 103.939]

---
## Step 26 — prepare iterators

In [ ]:
train_it = datagen.flow(trainX, trainY, batch_size=128)
	test_it = datagen.flow(testX, testY, batch_size=128)

---
## Step 27 — define model

In [ ]:
model = define_model()

---
## Step 28 — fit model

In [ ]:
history = model.fit_generator(train_it, steps_per_epoch=len(train_it),
		validation_data=test_it, validation_steps=len(test_it), epochs=20, verbose=0)

---
## Step 29 — evaluate model

In [ ]:
loss, fbeta = model.evaluate_generator(test_it, steps=len(test_it), verbose=0)
	print('> loss=%.3f, fbeta=%.3f' % (loss, fbeta))

---
## Step 30 — learning curves

In [ ]:
summarize_diagnostics(history)

---
## Step 31 — entry point, run the test harness

In [ ]:
run_test_harness()

---
## Learning Notes / 学习笔记

- **概念**: vgg16 transfer learning on the planet dataset with some trainable layers 是机器学习中的常用技术。  
  *vgg16 transfer learning on the planet dataset with some trainable layers is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Model Pretrained Trainable / 预训练模型
# Complete Code / 完整代码
# ===============================

# vgg16 transfer learning on the planet dataset with some trainable layers
import sys
from numpy import load
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from keras import backend
from keras.layers import Dense
from keras.layers import Flatten
from keras.optimizers import SGD
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.preprocessing.image import ImageDataGenerator

# load train and test dataset
def load_dataset():
	# load dataset
	data = load('planet_data.npz')
	X, y = data['arr_0'], data['arr_1']
	# separate into train and test datasets
	trainX, testX, trainY, testY = train_test_split(X, y, test_size=0.3, random_state=1)
	print(trainX.shape, trainY.shape, testX.shape, testY.shape)
	return trainX, trainY, testX, testY

# calculate fbeta score for multi-class/label classification
def fbeta(y_true, y_pred, beta=2):
	# clip predictions
	y_pred = backend.clip(y_pred, 0, 1)
	# calculate elements
	tp = backend.sum(backend.round(backend.clip(y_true * y_pred, 0, 1)), axis=1)
	fp = backend.sum(backend.round(backend.clip(y_pred - y_true, 0, 1)), axis=1)
	fn = backend.sum(backend.round(backend.clip(y_true - y_pred, 0, 1)), axis=1)
	# calculate precision
	p = tp / (tp + fp + backend.epsilon())
	# calculate recall
	r = tp / (tp + fn + backend.epsilon())
	# calculate fbeta, averaged across each class
	bb = beta ** 2
	fbeta_score = backend.mean((1 + bb) * (p * r) / (bb * p + r + backend.epsilon()))
	return fbeta_score

# define cnn model
def define_model(in_shape=(128, 128, 3), out_shape=17):
	# load model
	model = VGG16(include_top=False, input_shape=in_shape)
	# mark loaded layers as not trainable
	for layer in model.layers:
		layer.trainable = False
	# allow last vgg block to be trainable
	model.get_layer('block5_conv1').trainable = True
	model.get_layer('block5_conv2').trainable = True
	model.get_layer('block5_conv3').trainable = True
	model.get_layer('block5_pool').trainable = True
	# add new classifier layers
	flat1 = Flatten()(model.layers[-1].output)
	class1 = Dense(128, activation='relu', kernel_initializer='he_uniform')(flat1)
	output = Dense(out_shape, activation='sigmoid')(class1)
	# define new model
	model = Model(inputs=model.inputs, outputs=output)
	# compile model
	opt = SGD(lr=0.01, momentum=0.9)
	model.compile(optimizer=opt, loss='binary_crossentropy', metrics=[fbeta])
	return model

# plot diagnostic learning curves
def summarize_diagnostics(history):
	# plot loss
	pyplot.subplot(211)
	pyplot.title('Cross Entropy Loss')
	pyplot.plot(history.history['loss'], color='blue', label='train')
	pyplot.plot(history.history['val_loss'], color='orange', label='test')
	# plot accuracy
	pyplot.subplot(212)
	pyplot.title('Fbeta')
	pyplot.plot(history.history['fbeta'], color='blue', label='train')
	pyplot.plot(history.history['val_fbeta'], color='orange', label='test')
	# save plot to file
	filename = sys.argv[0].split('/')[-1]
	pyplot.savefig(filename + '_plot.png')
	pyplot.close()

# run the test harness for evaluating a model
def run_test_harness():
	# load dataset
	trainX, trainY, testX, testY = load_dataset()
	# create data generator
	datagen = ImageDataGenerator(featurewise_center=True)
	# specify imagenet mean values for centering
	datagen.mean = [123.68, 116.779, 103.939]
	# prepare iterators
	train_it = datagen.flow(trainX, trainY, batch_size=128)
	test_it = datagen.flow(testX, testY, batch_size=128)
	# define model
	model = define_model()
	# fit model
	history = model.fit_generator(train_it, steps_per_epoch=len(train_it),
		validation_data=test_it, validation_steps=len(test_it), epochs=20, verbose=0)
	# evaluate model
	loss, fbeta = model.evaluate_generator(test_it, steps=len(test_it), verbose=0)
	print('> loss=%.3f, fbeta=%.3f' % (loss, fbeta))
	# learning curves
	summarize_diagnostics(history)

# entry point, run the test harness
run_test_harness()

---

➡️ **Next / 下一步**: File 13 of 15